In [ ]:
import math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Global aesthetic settings
plt.rcParams["figure.dpi"] = 120  # Crisp resolution

sns.set_theme(
    style="ticks",
    font="sans-serif",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.bottom": True,
        "ytick.left": True,
        "xtick.major.size": 5,
        "ytick.major.size": 5,
        "xtick.major.width": 1,
        "ytick.major.width": 1,
        "xtick.direction": "out",
        "ytick.direction": "out",
    }
)

In [ ]:
# ============================================================
# Files to download from the drives data folder
# ============================================================
filenames = [
    "listings.csv.gz",
]

In [ ]:
# ============================================================
# Background logic for download needed once per Colab notebook
# ============================================================
from pathlib import Path
import io

import pandas as pd
from tqdm.auto import tqdm

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

drive_service = build("drive", "v3")

# Paste the Data folder ID printed by the previous upload script
DATA_FOLDER_ID = "15o0Cq15Uqy_oeWQ9egCcKVv2fi6W5Iu9"

LOCAL_DATA_DIR = Path("/content/data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)


def find_file_in_folder(filename: str, folder_id: str) -> str:
    query = (
        f"name = '{filename}' "
        f"and '{folder_id}' in parents "
        f"and trashed = false"
    )

    results = drive_service.files().list(
        q=query,
        fields="files(id, name, size)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()

    files = results.get("files", [])

    if not files:
        raise FileNotFoundError(f"Could not find {filename} in Drive data folder.")

    return files[0]["id"]


def download_drive_file(file_id: str, local_path: Path):
    request = drive_service.files().get_media(
        fileId=file_id,
        supportsAllDrives=True,
    )

    with open(local_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)

        done = False
        with tqdm(unit="B", unit_scale=True, desc=local_path.name) as progress:
            previous_bytes = 0

            while not done:
                status, done = downloader.next_chunk()

                if status:
                    current_bytes = int(status.resumable_progress)
                    progress.update(current_bytes - previous_bytes)
                    previous_bytes = current_bytes


# Download files from Google Drive/data to local Colab runtime
for filename in filenames:
    local_path = LOCAL_DATA_DIR / filename

    if local_path.exists():
        print(f"Skipping {filename}, already downloaded locally.")
        continue

    print(f"Finding {filename} in Drive...")
    file_id = find_file_in_folder(filename, DATA_FOLDER_ID)

    print(f"Downloading {filename} to {local_path}...")
    download_drive_file(file_id, local_path)

print("\nFiles available locally in:")
print(LOCAL_DATA_DIR)
!ls "data"

Finding listings.csv.gz in Drive...


listings.csv.gz: 0.00B [00:00, ?B/s]


Files available locally in:
/content/data
listings.csv.gz


In [ ]:
# ============================================================
# Background logic for upload needed once per Colab notebook
# ============================================================

from pathlib import Path
import mimetypes

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload


# Authenticate Colab with Google Drive
auth.authenticate_user()

# Global Drive service used by the upload function
drive_service = build("drive", "v3")


# Your notebook file ID
NOTEBOOK_FILE_ID = "15PQoSKwHdwrxRwMaaZS6LPdNZAVFKnUT"


def escape_drive_query_string(s: str) -> str:
    """
    Escape a string for use inside a Google Drive API query.
    Needed for filenames/folder names containing apostrophes or backslashes.
    """
    return s.replace("\\", "\\\\").replace("'", "\\'")


def get_notebook_parent_folder_id(notebook_file_id: str) -> str:
    """
    Return the Drive folder ID containing the current notebook.
    """
    notebook_meta = drive_service.files().get(
        fileId=notebook_file_id,
        fields="id, name, parents",
        supportsAllDrives=True,
    ).execute()

    parents = notebook_meta.get("parents", [])

    if not parents:
        raise RuntimeError(
            "Could not find the notebook's parent folder. "
            "This usually means the notebook is not stored normally in Drive."
        )

    return parents[0]


def get_or_create_folder(folder_name: str, parent_id: str) -> str:
    """
    Return the ID of a folder named `folder_name` inside `parent_id`.
    Create it if it does not exist.
    """
    safe_name = escape_drive_query_string(folder_name)

    query = (
        f"name = '{safe_name}' "
        f"and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents "
        f"and trashed = false"
    )

    results = drive_service.files().list(
        q=query,
        fields="files(id, name)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()

    folders = results.get("files", [])

    if folders:
        return folders[0]["id"]

    metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id],
    }

    folder = drive_service.files().create(
        body=metadata,
        fields="id",
        supportsAllDrives=True,
    ).execute()

    return folder["id"]


# Global destination folder used by the upload function
PARENT_FOLDER_ID = get_notebook_parent_folder_id(NOTEBOOK_FILE_ID)
DATA_FOLDER_ID = get_or_create_folder("data", PARENT_FOLDER_ID)

print("Parent folder ID:", PARENT_FOLDER_ID)
print("Data folder ID:", DATA_FOLDER_ID)

Parent folder ID: 1-0XXChZ8L1Qsl7-c3655LFoDgxWX-c3f
Data folder ID: 15o0Cq15Uqy_oeWQ9egCcKVv2fi6W5Iu9


In [ ]:
# ============================================================
# Function to upload dataframes with
# ============================================================
import io
from googleapiclient.http import MediaIoBaseUpload


def upload_dataframe_as_csv(df, filename: str, index: bool = False) -> dict:
    """
    Convert a local pandas DataFrame to CSV and upload it to Drive/data.

    Uses global variables:
        - drive_service
        - DATA_FOLDER_ID
        - escape_drive_query_string

    Parameters
    ----------
    df:
        pandas DataFrame to upload.
    filename:
        Target filename in Google Drive, e.g. "listings_cleaned.csv".
        If ".csv" is missing, it is added automatically.
    index:
        Whether to include the DataFrame index in the CSV.

    Returns
    -------
    dict:
        {
            "action": "created" | "updated",
            "file_id": "...",
            "name": "..."
        }
    """

    if not filename.endswith(".csv"):
        filename += ".csv"

    safe_filename = escape_drive_query_string(filename)

    query = (
        f"name = '{safe_filename}' "
        f"and '{DATA_FOLDER_ID}' in parents "
        f"and trashed = false"
    )

    results = drive_service.files().list(
        q=query,
        fields="files(id, name)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()

    existing_files = results.get("files", [])

    csv_bytes = df.to_csv(index=index).encode("utf-8")
    csv_buffer = io.BytesIO(csv_bytes)

    media = MediaIoBaseUpload(
        csv_buffer,
        mimetype="text/csv",
        resumable=True,
    )

    if existing_files:
        file_id = existing_files[0]["id"]

        print(f"Updating existing CSV in Drive/data: {filename}")

        updated_file = drive_service.files().update(
            fileId=file_id,
            media_body=media,
            fields="id, name",
            supportsAllDrives=True,
        ).execute()

        return {
            "action": "updated",
            "file_id": updated_file["id"],
            "name": updated_file["name"],
        }

    print(f"Uploading new CSV to Drive/data: {filename}")

    metadata = {
        "name": filename,
        "parents": [DATA_FOLDER_ID],
        "mimeType": "text/csv",
    }

    created_file = drive_service.files().create(
        body=metadata,
        media_body=media,
        fields="id, name",
        supportsAllDrives=True,
    ).execute()

    return {
        "action": "created",
        "file_id": created_file["id"],
        "name": created_file["name"],
    }

In [ ]:
!ls data/

listings.csv.gz


In [ ]:
df_raw = pd.read_csv("data/listings.csv.gz")
df_raw.columns

/tmp/ipykernel_1048/1734377795.py:1: DtypeWarning: Columns (72) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("data/listings.csv.gz")


Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [ ]:
# All rows have the same last_scraped date, so last_scraped itself contains
# no cross-sectional predictive information and should not be a model feature.
#
# It is still useful as a fixed reference date when constructing temporal
# features such as host_tenure_days, days_since_first_review, and
# days_since_last_review.
#
# We excluded all constructed features that directly or indirectly use price,
# such as estimated_revenue_l365d. Including them would leak the target.

# Structural and location features
structural_features = [
    "latitude",
    "longitude",

    # Preferred cleaned neighbourhood variable.
    # neighbourhood_group_cleansed is excluded because it contains no information.
    "neighbourhood_cleansed",

    "property_type",
    "room_type",

    "accommodates",

    # bathrooms and bathrooms_text should be parsed together.
    # Possible engineered features:
    # - bathroom_count
    # - bathroom_is_shared
    # - bathroom_is_private
    # - bathroom_information_missing
    "bathrooms",
    "bathrooms_text",

    # Convert into features such as:
    # - has_license
    # - possibly license_type, if meaningful categories can be extracted
    "license",

    # Add explicit missing-value indicators.
    "bedrooms",
    "beds",

    # Parse raw amenities representation instead of treating the complete
    # string as one categorical value.
    # Possible engineered features:
    # - amenity_count
    # - has_wifi
    # - has_kitchen
    # - has_parking
    # - has_washer
    # - has_air_conditioning
    # - has_workspace
    # - has_balcony, if consistently identifiable
    "amenities",
]

host_features = [
    # Only about 4 values are missing.
    # Convert to:
    # host_tenure_days = fixed_last_scraped_date - host_since
    #
    # Since last_scraped is constant, the same fixed date is used for every row.
    "host_since",

    # Approximately 1/4 of the dataset is missing this value.
    #
    # Possible treatment:
    # - explicit "MISSING" category
    # - or host_location_missing indicator
    # - engineered host_is_local feature when location can be parsed
    "host_location",

    # Parse into verification indicators or a verification count rather than
    # treating each complete list/string as a separate category.
    "host_verifications",

    # Preserve missing values as "unknown"; in has profile pic there are only ['t', nan]
    "host_has_profile_pic",
    "host_identity_verified",
]

# ChatGPT:
# Do not include all of these without checking redundancy.
#
# host_listings_count is host-reported, whereas calculated_host_listings_count
# and its components are calculated from the observed Airbnb data.
#
# Compare correlations and definitions before deciding whether to retain all
# raw counts. Tree models can tolerate correlated variables, but correlated
# features make importance estimates less stable and harder to interpret.
host_scale_features = [
    "host_listings_count",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
]

review_features = [
    # Convert dates into relative features using last_scraped date:
    # - days_since_first_review
    # - days_since_last_review
    #
    # Should add has_reviews and missing-date indicators where appropriate.
    "first_review",
    "last_review",

    # Count variables are typically right-skewed.
    # Many listings have 0-20 reviews
    # Fewer have 50-100
    # Using log1p transformations makes values better for training, expecially for linear models.
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",

    # Reviews-per-month and the review-score variables have:
    # - total rows: 8274
    # - missing rows: 1916
    # - missing fraction: approximately 23%
    #
    # Missingness likely identifies listings with no
    # reviews or insufficient review history and may itself predict price.
    #
    # - add a missing-value indicator
    # - median-impute score variables inside each training fold (calculating on whole training would leak data)
    # - set reviews_per_month to 0 only when number_of_reviews == 0
    # - otherwise impute reviews_per_month and preserve its missing indicator
    "reviews_per_month",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
]

availibility_features = [
    # Total rows: 8274
    # Missing rows: 334
    #
    # Observed values are only {"t", NaN}; there is no explicit "f".
    #
    # Idea:
    # - has_availability_reported = 1 when value == "t", otherwise 0
    # - has_availability_missing = 1 when the raw value is NaN
    "has_availability",

    # These columns are complete, despite has_availability containing NaNs.
    # Should check whether their values remain meaningful for rows where
    # has_availability is missing.
    #
    # low availability could indicate bookings,
    # owner-blocked dates, or incomplete calendar information.
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "availability_eoy",
]

booking_features = [
    # Should check distributions for extreme or placeholder maximum-night values.
    # A log1p transformation may be useful for linear models.
    "maximum_nights",
    "minimum_nights",

    # These summarize near-term calendar restrictions and may overlap strongly
    # with the raw minimum_nights and maximum_nights variables.
    # Check redundancy through correlations and feature ablations.
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm",

    # Should encode as a boolean/categorical feature while preserving unknown values,
    # should any exist.
    "instant_bookable",
]

# Many columns in this group contain substantial missingness.
#
# Low-cost engineered features:
# - text_missing
# - character_count
# - word_count
# - number of exclamation marks
#
# host_about is approximately 60% missing. A language model or separate text
# model could extract richer quality information, but that is outside the
# current time budget.
#
# host_neighbourhood is not included anywhere because approximately 3/4 of it
# is missing and the listing's own location variables are more relevant.
text_features = [
    "name",
    "description",
    "neighborhood_overview",
    "host_about",
]

# Some major outliers are seen in the price data.
# Should check if they cover different categories such as luxary hotels etc.
# log1p price could be useful to have larger values have less impact
target = "price"

In [ ]:
relevant_columns = [
  'id',
 'review_scores_rating',
 'number_of_reviews',
 'name',
 'description',
 'neighborhood_overview',
 'host_since',
 'host_response_time',
 'host_response_rate',
 'latitude',
 'longitude',
 'bathrooms',
 'bedrooms',
 'beds',
 'price',
 'minimum_nights',
 'maximum_nights',
]
df_relevant = df_raw[relevant_columns]

In [ ]:
df = df_relevant[df_relevant["price"].notna()].copy()
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [ ]:
# Adding distance km
# The haversine distance in km between two points (lat1, lon1) (lat2, lon2)
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

#Munich city center (Marienplatz) at (lat: 48.13737881325798, long: 11.575437772781985)
#Predict price of Airbnb based on distance to center
center_lat = 48.13737881325798
center_lon = 11.575437772781985

df["distance_km"] = haversine_km(
    center_lat,
    center_lon,
    df["latitude"],
    df["longitude"]
)
df

,id,review_scores_rating,number_of_reviews,name,description,neighborhood_overview,host_since,host_response_time,host_response_rate,latitude,longitude,bathrooms,bedrooms,beds,price,minimum_nights,maximum_nights,distance_km
0,97945,4.84,126,Deluxw-Apartm. with roof terrace,NaN,We are living in a outskirt of Munich its call...,2011-04-18,NaN,NaN,48.114920,11.489540,1.0,1.0,1.0,105.0,2,90,6.847144
2,127383,4.84,119,City apartment next to Pinakothek,NaN,NaN,2011-05-26,within a few hours,100%,48.151990,11.564820,1.0,1.0,1.0,288.0,3,20,1.805609
4,170154,4.96,590,"Own floor & bath, parking & breakfast","Enjoy a quiet neighbourhood, easy access to th...",NaN,2010-04-14,within a few hours,100%,48.108140,11.527330,1.0,1.0,2.0,142.0,3,1125,4.829239
6,172977,4.83,120,Beautiful room close to centre,NaN,You will find in a very close walking distance...,2011-07-16,within a day,80%,48.149570,11.555070,1.0,1.0,1.0,120.0,2,7,2.030133
7,187359,4.90,53,Charming apartment near City Center,My apartment is located in the heart of Munich...,The best bakers of Munich are right next door....,2011-08-01,NaN,NaN,48.156380,11.539350,1.0,1.0,1.0,68.0,2,35,3.410666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8269,1518203607995782888,NaN,0,Modernes City-Apartment I S-Bahn,Experience Munich at its best! Our stylish apa...,NaN,2014-01-10,within an hour,100%,48.145860,11.518510,1.0,2.0,5.0,427.0,1,365,4.328005
8270,1518242443990757686,NaN,0,„Wiesn-Bleibe“ für Jedermann,Charming apartment in the heart of Munich – on...,NaN,2025-09-25,NaN,NaN,48.129810,11.575270,1.0,1.0,1.0,152.0,1,9,0.841706
8271,1518254619449006549,NaN,0,Kleines gemütliches Zimmer 5min zur Theresienw...,NaN,NaN,2017-07-13,NaN,NaN,48.123600,11.551410,1.0,1.0,1.0,200.0,1,3,2.351034
8272,1518301717394626264,NaN,0,Ein Zimmer mitten in Altschwabing!,Enjoy a stylish experience at this centrally-l...,NaN,2022-10-16,NaN,NaN,48.160853,11.586585,0.5,1.0,1.0,120.0,1,365,2.738060


In [ ]:
p99 = df["price"].quantile(0.998)
df_p99 = df[df["price"] <= p99]
df_p99.columns

Index(['id', 'review_scores_rating', 'number_of_reviews', 'name',
       'description', 'neighborhood_overview', 'host_since',
       'host_response_time', 'host_response_rate', 'latitude', 'longitude',
       'bathrooms', 'bedrooms', 'beds', 'price', 'minimum_nights',
       'maximum_nights', 'distance_km'],
      dtype='object')

In [ ]:
# Safe cleaned data to csv files,
# df_p99 -> listings_relevant_clean_price_p99
# df -> listings_relevant_clean_price

In [ ]:
upload_dataframe_as_csv(df_p99, "listings_relevant_clean_price_p99.csv")
upload_dataframe_as_csv(df, "listings_relevant_clean_price.csv")

Updating existing CSV in Drive/data: listings_relevant_clean_price_p99.csv
Updating existing CSV in Drive/data: listings_relevant_clean_price.csv


{'action': 'updated',
 'file_id': '1X3I4dBPknb2hH1KSnKpuD-mKHggkZxTV',
 'name': 'listings_relevant_clean_price.csv'}